# Bitwise XOR with native FlagQuantum

Write arithmetic over named quantum values; the compiler constructs the gates.
This Notebook preserves both examples and their order from the upstream tutorial.
It runs locally on CPU. Seeded sampling from executed exact probabilities replaces the upstream cloud sampler; no hardware result is claimed.

`^` acts bit by bit. Unequal widths use sign extension for signed words and zero extension for unsigned words. The examples below use the original positive inputs; they do not demonstrate general signed arithmetic.


In [ ]:
from collections import Counter
from random import Random
from typing import Annotated

import flagquantum as fq
from flagquantum.synth import Output, QNum, allocate, qfunc, synthesize


## 1. Two quantum variables

The left operand is a five-bit signed integer; the right is a three-bit unsigned integer. Their values are 4 and 5. The result is `4 ^ 5 = 1`. Explicit annotations carry the original widths and signedness; no wire numbers are required.


In [ ]:
@qfunc
def register_operation(
    a: Output[Annotated[QNum, (5, True, 0)]],
    b: Output[Annotated[QNum, 3]],
    res: Output[QNum],
) -> None:
    allocate(a)
    allocate(b)
    a ^= 4
    b ^= 5
    res |= a ^ b


In [ ]:
register_program = synthesize(register_operation)
register_result = fq.run(register_program)
register_probabilities = dict(register_program.numeric_probabilities(
    register_result, registers=("a", "b", "res")
))
assert register_probabilities == {(4, 5, 1): 1.0}
register_samples = Counter(Random(17).choices(
    list(register_probabilities), weights=list(register_probabilities.values()), k=1000
))
assert register_samples == {(4, 5, 1): 1000}
print("Probabilities:", register_probabilities)
print("Counts:", dict(register_samples))
print("Resources:", register_program.report["resources"])


## 2. Integer and quantum variable

Keep the original literal 3 and quantum value 4: `3 ^ 4 = 7`. `a |= 4` creates a fresh value, while the operation in the result expression computes the bitwise XOR. Named reads avoid manual bit-string decoding.


In [ ]:
@qfunc
def literal_operation(a: Output[QNum], res: Output[QNum]) -> None:
    a |= 4
    res |= 3 ^ a


In [ ]:
literal_program = synthesize(literal_operation)
literal_result = fq.run(literal_program)
literal_probabilities = dict(literal_program.probabilities(
    literal_result, registers=("a", "res")
))
assert literal_probabilities == {(4, 7): 1.0}
literal_samples = Counter(Random(17).choices(
    list(literal_probabilities), weights=list(literal_probabilities.values()), k=1000
))
assert literal_samples == {(4, 7): 1000}
print("Probabilities:", literal_probabilities)
print("Counts:", dict(literal_samples))
print("Resources:", literal_program.report["resources"])


## Interpretation and validation

Both inputs are basis states, so every shot has the same named outcome. This is expected, not evidence of a speedup. Probability agreement alone cannot detect a global phase error: the acceptance tests separately compare the complete complex state with independent Circuit references and check zero hidden workspace.

The companion [workflow guide](WORKFLOW.md) records source-cell coverage and local execution limits.
